In [1]:
import pandas as pd
import duckdb

# make sure to replace 'file.parquet' with your file path
df = pd.read_parquet('./data/idMinMaxDepth.parquet')
# df = df.head(10)

# use the values in the dataset_id to search via
# duckdb for the @id to generate the JSON-LD with.

def search_duckdb(x):
    x = duckdb.sql(f"SELECT url FROM read_json('./jsonld/obis_source/*.jsonld') WHERE url like '%{x}%'").fetchall()
    xs = [str(item[0]) for item in x]
    return(xs)

df['docid'] = df['dataset_id'].apply(lambda x: search_duckdb(x))

dfe = df.explode('docid')


In [2]:
dfe

,dataset_id,min_depth,max_depth,docid
0,0ec403c9-23b3-4529-ae51-8a0877f22450,6.0,63.0,https://obis.org/dataset/0ec403c9-23b3-4529-ae...
0,0ec403c9-23b3-4529-ae51-8a0877f22450,6.0,63.0,https://obis.org/dataset/0ec403c9-23b3-4529-ae...
1,d1001fdc-d4a2-4d01-bcd2-2e122f2ca537,-9.0,7250.0,https://obis.org/dataset/d1001fdc-d4a2-4d01-bc...
1,d1001fdc-d4a2-4d01-bcd2-2e122f2ca537,-9.0,7250.0,https://obis.org/dataset/d1001fdc-d4a2-4d01-bc...
2,3cd94a39-6a58-4f0e-9c74-e044e7a0e737,0.0,10000.0,https://obis.org/dataset/3cd94a39-6a58-4f0e-9c...
...,...,...,...,...
4873,708bb02c-e0bb-408f-a867-92f0bc2f70ef,NaN,NaN,https://obis.org/dataset/708bb02c-e0bb-408f-a8...
4873,708bb02c-e0bb-408f-a867-92f0bc2f70ef,NaN,NaN,https://obis.org/dataset/708bb02c-e0bb-408f-a8...
4874,9362fac0-8252-4afe-9855-822113401709,NaN,NaN,https://obis.org/dataset/9362fac0-8252-4afe-98...
4874,9362fac0-8252-4afe-9855-822113401709,NaN,NaN,https://obis.org/dataset/9362fac0-8252-4afe-98...


In [5]:
# remove where columns min/max have Nan
dfe_strict = dfe.dropna(subset=['max_depth', 'min_depth'], how='any')

In [6]:
dfe_strict

,dataset_id,min_depth,max_depth,docid
0,0ec403c9-23b3-4529-ae51-8a0877f22450,6.00,63.000000,https://obis.org/dataset/0ec403c9-23b3-4529-ae...
0,0ec403c9-23b3-4529-ae51-8a0877f22450,6.00,63.000000,https://obis.org/dataset/0ec403c9-23b3-4529-ae...
1,d1001fdc-d4a2-4d01-bcd2-2e122f2ca537,-9.00,7250.000000,https://obis.org/dataset/d1001fdc-d4a2-4d01-bc...
1,d1001fdc-d4a2-4d01-bcd2-2e122f2ca537,-9.00,7250.000000,https://obis.org/dataset/d1001fdc-d4a2-4d01-bc...
2,3cd94a39-6a58-4f0e-9c74-e044e7a0e737,0.00,10000.000000,https://obis.org/dataset/3cd94a39-6a58-4f0e-9c...
...,...,...,...,...
4848,08a645b0-1947-4168-b628-405cdb8e54fc,0.00,0.000000,https://obis.org/dataset/08a645b0-1947-4168-b6...
4848,08a645b0-1947-4168-b628-405cdb8e54fc,0.00,0.000000,https://obis.org/dataset/08a645b0-1947-4168-b6...
4862,5cefe6a4-68f1-43c9-8f1b-4d857454f1c2,0.00,122.000000,https://obis.org/dataset/5cefe6a4-68f1-43c9-8f...
4868,59ea7e58-052b-4107-9b7d-23ec2c46122e,0.46,375.829987,https://obis.org/dataset/59ea7e58-052b-4107-9b...


In [11]:
def populate_template(row):
    template = """ {{
      "@context": {{
        "@vocab": "https://schema.org/"
      }},
      "@id": "{docid}",
      "@type": "Dataset",
      "variableMeasured": [
        {{
          "@type": "PropertyValue",
          "name": "depth",
          "description": "Parsed and validated by OBIS.",
          "minValue": "{MIN}",
          "maxValue": "{MAX}",
          "propertyID": "https://obis.org/data/access/",
          "measurementTechnique": "Parsed and validated by OBIS.",
          "unitText": "m",
          "unitCode": [
            "https://qudt.org/vocab/unit/M", "https://vocab.nerc.ac.uk/collection/P06/current/ULAA/",
            "http://dbpedia.org/resource/Metre"
          ]
        }}
      ]
    }}
    """
    
    return template.format(MAX=row['max_depth'], MIN=row['min_depth'],  docid=row['docid'])




### All elements, where the min or max might be NONE

In [13]:
dfe = dfe.assign(jsonld=dfe.apply(populate_template, axis=1))


In [15]:

for index, row in dfe.iterrows():
    filename = str('./jsonld/output_all/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])

### Only elements where the min and max have values


In [16]:
dfe_strict = dfe_strict.assign(jsonld=dfe_strict.apply(populate_template, axis=1))


In [17]:
for index, row in dfe_strict.iterrows():
    filename = str('./jsonld/output_strict/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])